# **Validación y explotación del modelo Corepulse**

Este notebook documenta el proceso posterior a la preparación del modelo dimensional del proyecto **Corepulse Sales Analytics**.

La fase anterior dejó preparados los scripts de transformación, las tablas finales del modelo y el script SQL de carga en Snowflake. En esta fase se valida que el modelo se haya cargado correctamente en base de datos y se preparan las bases para su explotación en herramientas de Business Intelligence.


## **1. Objetivo de esta fase**.

El objetivo principal de esta fase es comprobar que el modelo estrella cargado en Snowflake es consistente y está listo para ser conectado a herramientas como Power BI o Looker Studio.

Las validaciones se centran en:

- Comprobar que las tablas se han creado y cargado correctamente.
- Validar que las dimensiones no tienen claves duplicadas.
- Comprobar que la tabla de hechos no contiene claves huérfanas.
- Validar que la granularidad de la fact se mantiene correctamente.
- Dejar documentado el estado del modelo antes de construir dashboards.


---

## **2. Modelo cargado en Snowflake**.

El modelo sigue una estructura de **modelo estrella**, con una tabla de hechos central y varias dimensiones descriptivas.

### Tabla de hechos

- `fact_ventas`

### Dimensiones

- `dim_producto`
- `dim_categoria`
- `dim_proveedor`
- `dim_calendario`

La granularidad esperada de la fact es:

```text
1 fila = 1 producto + 1 semana de negocio
```

Esto significa que la combinación `product_id + year_week_key` debe ser única dentro de `fact_ventas`.


----

## **3. Validación de los datos cargados.**

### **3.1. Validaciones: conteo de filas por tabla.**

La primera comprobación sirve para confirmar que todas las tablas se han cargado y que el volumen de registros es coherente con lo esperado.

Esta validación no comprueba todavía relaciones ni duplicados; simplemente responde a la pregunta:

> ¿Se han creado y cargado las tablas principales del modelo?


```yaml
-- Conteo de filas por tabla
SELECT 'dim_categoria' AS tabla, COUNT(*) AS filas FROM dim_categoria
UNION ALL
SELECT 'dim_proveedor', COUNT(*) FROM dim_proveedor
UNION ALL
SELECT 'dim_producto', COUNT(*) FROM dim_producto
UNION ALL
SELECT 'dim_calendario', COUNT(*) FROM dim_calendario
UNION ALL
SELECT 'fact_ventas', COUNT(*) FROM fact_ventas;

### Resultado obtenido

| Tabla | Filas |
|---|---:|
| dim_categoria | 19 |
| dim_proveedor | 20 |
| dim_producto | 60 |
| dim_calendario | 159 |
| fact_ventas | 9540 |

### Interpretación

El modelo se ha cargado correctamente en Snowflake. La fact contiene 9540 registros, coherentes con una estructura semanal por producto.

La lectura de la fact es coherente con el grano esperado:

```text
60 productos × 159 semanas = 9540 filas
```

Por tanto, a nivel de volumen, la carga parece correcta.


### **3.2. Validación 2: duplicados en dimensiones.**

Las dimensiones deben tener una fila única por cada clave primaria lógica.

Aunque en Snowflake se pueden declarar claves primarias y foráneas, en tablas estándar estas restricciones funcionan principalmente como metadatos. Por eso es importante comprobar manualmente que las claves realmente se comportan como únicas.

En el caso de `dim_calendario`, cada `year_week_key` debe aparecer una sola vez.


```yaml

-- Duplicados en dim_calendario
SELECT 
    year_week_key,
    COUNT(*) AS n_filas
FROM dim_calendario
GROUP BY year_week_key
HAVING COUNT(*) > 1;


### Lógica de la query

- `GROUP BY year_week_key` agrupa todas las filas que pertenecen a la misma semana.
- `COUNT(*)` cuenta cuántas filas hay para cada semana.
- `HAVING COUNT(*) > 1` muestra únicamente las semanas que aparecen más de una vez.

La diferencia clave es:

```text
WHERE  → filtra filas antes de agrupar
HAVING → filtra grupos después de agrupar
```

En este caso se usa `HAVING` porque queremos filtrar grupos agregados, no filas individuales.

### Resultado obtenido

La consulta devuelve **0 filas**.

### Interpretación

No hay semanas duplicadas en `dim_calendario`. La clave `year_week_key` funciona correctamente como identificador único de semana.


### **3.3. Validación 3: claves huérfanas entre fact y calendario.**

Esta validación comprueba si existen registros en `fact_ventas` cuya semana no exista en `dim_calendario`.

Una clave huérfana aparecería si la fact contiene un `year_week_key` que no tiene correspondencia en la dimensión calendario.


```yaml

-- Claves huérfanas entre fact_ventas y dim_calendario
SELECT COUNT(*) AS fact_sin_calendario
FROM fact_ventas f
LEFT JOIN dim_calendario c
    ON f.year_week_key = c.year_week_key
WHERE c.year_week_key IS NULL;

### Lógica de la query

Se utiliza un `LEFT JOIN` desde `fact_ventas` hacia `dim_calendario`.

Esto conserva todas las filas de la fact. Si una fila de la fact no encuentra coincidencia en calendario, las columnas de `dim_calendario` quedan como `NULL`.

Por eso se filtra con:

```sql
WHERE c.year_week_key IS NULL
```

Esa condición identifica las filas de la fact que no han encontrado correspondencia en la dimensión.

### Resultado obtenido

`fact_sin_calendario = 0`

### Interpretación

Todas las semanas presentes en `fact_ventas` existen también en `dim_calendario`. No hay claves huérfanas en la relación temporal.


### **3.4. Validación 4: granularidad de la fact.**

La granularidad esperada de `fact_ventas` es:

```text
1 fila = 1 producto + 1 semana de negocio
```

Por tanto, no debe existir más de una fila para la misma combinación de `product_id` y `year_week_key`.


```yaml

-- Validación de granularidad de la fact
SELECT 
    product_id,
    year_week_key,
    COUNT(*) AS n_filas
FROM fact_ventas
GROUP BY product_id, year_week_key
HAVING COUNT(*) > 1;

### Lógica de la query

- `GROUP BY product_id, year_week_key` agrupa por cada combinación producto-semana.
- `COUNT(*)` cuenta cuántas filas existen para cada combinación.
- `HAVING COUNT(*) > 1` filtra únicamente las combinaciones repetidas.

Si esta query devolviera resultados, significaría que la fact tiene duplicados a nivel de grano.

### Resultado obtenido

La consulta devuelve **0 filas**.

### Interpretación

La fact respeta correctamente la granularidad definida. No existen duplicados por producto y semana.


### **3.5. Resumen de validaciones.**

| Validación | Resultado | Interpretación |
|---|---:|---|
| Conteo de tablas | Correcto | Las tablas se han cargado correctamente. |
| Duplicados en calendario | 0 filas | `year_week_key` es único en `dim_calendario`. |
| Fact sin calendario | 0 | Todas las semanas de la fact existen en calendario. |
| Duplicados producto-semana | 0 filas | La fact respeta su granularidad. |

Conclusión: el modelo cargado en Snowflake está correctamente estructurado y puede utilizarse como base para la fase de visualización.


----

## 8. Próximos pasos

Una vez validado el modelo en Snowflake, los siguientes pasos del proyecto son:

1. Conectar Snowflake con Power BI.
2. Revisar relaciones del modelo en Power BI.
3. Crear medidas DAX principales.
4. Diseñar las páginas del dashboard.
5. Replicar o adaptar el análisis en Looker Studio.
6. Documentar decisiones de diseño y visualización.

Este notebook se actualizará progresivamente conforme avance la fase de explotación analítica.
